# Week12 Lab —  Q-Learning with FrozenLake


## 0. Install and Import Libraries
- `gymnasium`: provides the FrozenLake environment.
- `numpy`: stores and updates the Q-table.
- `matplotlib`: plots training results.

In [7]:
# If needed, uncomment the following line:
# !pip install gymnasium

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

## Introduction to ~OpenAI gym~ Gymnasium
In this notebook we will be using [gymnasium](https://github.com/Farama-Foundation/Gymnasium), a great toolkit for developing and comparing Reinforcement Learning algorithms. It provides many environments for your learning *agents* to interact with. 

If Gymnasium is not installed, run:
```python
!pip install gymnasium
```

**Tip**: `gym.envs.registry` is a dictionary containing all available environments

In [8]:
envs = gym.envs.registry
sorted(envs.keys())[:15] + ["..."]

['Acrobot-v1',
 'Ant-v2',
 'Ant-v3',
 'Ant-v4',
 'Ant-v5',
 'BipedalWalker-v3',
 'BipedalWalkerHardcore-v3',
 'Blackjack-v1',
 'CarRacing-v3',
 'CartPole-v0',
 'CartPole-v1',
 'CliffWalking-v1',
 'CliffWalkingSlippery-v1',
 'FrozenLake-v1',
 'FrozenLake8x8-v1',
 '...']

In this lab, let's use Q-learning to train an agent to play FrozenLake.



## 1. Load FrozenLake Environment

You can check the documentation for FrozenLake here:  
https://gymnasium.farama.org/environments/toy_text/frozen_lake/

*Note: We use is_slippery=False as a demo* 
```python
is_slippery=False
```
This makes the environment deterministic. If the agent chooses right, it really moves right.  

Later, you can set `is_slippery=True` to show stochastic transitions,meaning the agent’s movement is uncertain.


In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=False)
#env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="human")
#shows the specification for the CartPole-v1 environment


num_states = env.observation_space.n
num_actions = env.action_space.n

# Action names for readability.
action_names = {
    0: "Left",
    1: "Down",
    2: "Right",
    3: "Up"
}

possible_actions = [np.arange(num_actions) for state in range(num_states)]

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
for action_id, action_name in action_names.items():
    print(action_id, "=", action_name)


Observation space: Discrete(16)
Action space: Discrete(4)
0 = Left
1 = Down
2 = Right
3 = Up


## 2. Set Hyperparameters

1. Initial Learning Rate: ```alpha = 0.05```
2. Discount Factor: ```gamma```
- \(\gamma=0\): only care about immediate reward.
- \(\gamma\approx1\): care strongly about future reward.
3. ```epsilon``` controls exploration, the agent will chooses a random action ```epsilon``` of the time; and chooses the action with the highest Q-value ```1-epsilon``` of the time.

In [79]:
num_episodes = 5000  
alpha = 0.2      
gamma = 0.95 
epsilon = 0.5     # initial exploration rate

print("maximum episode steps:", env.spec.max_episode_steps) 

maximum episode steps: 100


## 3. The Q-Learning Update Formula

$$Target = r+\gamma\max_{a'}Q(s',a')$$

$$Q(s,a) \leftarrow Q(s,a)+\alpha\left[Target-Q(s,a)\right]$$

Where:
- $Q(s,a)$: old estimate
- $r$: immediate reward
- $s'$: next state
- $max_{a'}Q(s',a')$: best estimated future value from the next state
- $alpha$: learning rate
- $gamma$: discount factor




In [80]:
def update_Q_values(Q_values, state, action, reward, next_state, alpha, gamma):
    next_value = np.max(Q_values[next_state])

    Target =  reward + gamma * next_value
    Q_values[state, action] = Q_values[state, action] - alpha * (Q_values[state, action]-Target) 

    return Q_values

## 4. Train the Agent

1. At the beginning, the agent knows nothing. So we initialize every Q-value to 0. 
2. The exploratory behavior policy based on epsilon
3. When taking a new step, we can get reward or next_sate by call function: 
    ```next_state, reward, terminated, truncated, info = env.step(action)```
    1. terminated = True (state = Hole or state = Goal) 
    2. truncated means the episode was stopped by an outside limit

4. For FrozenLake, an episode ends when the agent reaches a hole or the goal. When this happens, we reset back to the start state so training can continue.
We also record:
- return per episode
- steps per episode


In [81]:
# Initialize Q-table before training
Q_values = np.zeros((env.observation_space.n, env.action_space.n))

episode_returns = [] # Store total reward for each episode.
episode_steps = []   # Store number of steps for each episode.
success_rate = []# Store success indicator for each episode.


# Run the Q-learning algorithm.
for iteration in range(num_episodes):
    state,info = env.reset()
    done = False
    episode_return = 0
    episode_step = 0

    for step in range(env.spec.max_episode_steps): #100 by default
        
        # 1. Choose action with Epsilon-greedy action selection
        if np.random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(Q_values[state]))

        # 2. Take one step and observe next state, reward, and done.
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        if terminated:
            target = reward
        else:
            target = reward + gamma * np.max(Q_values[next_state])
        
        # 3. Update Q-table, using the textbook-style update
        Q_values = update_Q_values(Q_values, state, action, reward, next_state, alpha, gamma)
        state = next_state

        # 4. Track episode metrics
        episode_return += reward
        episode_step += 1
    
        # If the episode ended, reset to the start state for the next episode.
        if done:
            episode_returns.append(episode_return)
            episode_steps.append(episode_step)
            success_rate.append(int(episode_return > 0))
            break
        
    # Print progress every 1000 episodes.
    if (iteration + 1) % 1000 == 0:
        recent_success = np.mean(success_rate[-1000:]) * 100
        print("Episode:", iteration + 1,
              "| Reward:", episode_return,
              "| Steps:", episode_step,
              "| Recent success rate:", f"{recent_success:.1f}%")

print("Training finished.")
print("Number of completed episodes:", len(episode_returns))

Episode: 1000 | Reward: 0 | Steps: 8 | Recent success rate: 25.2%
Episode: 2000 | Reward: 0 | Steps: 4 | Recent success rate: 43.9%
Episode: 3000 | Reward: 1 | Steps: 10 | Recent success rate: 43.5%
Episode: 4000 | Reward: 1 | Steps: 7 | Recent success rate: 41.4%
Episode: 5000 | Reward: 0 | Steps: 7 | Recent success rate: 41.2%
Training finished.
Number of completed episodes: 5000


## 5. Inspect the Learned Q-Table

After training, the Q-table contains learned estimates of long-term action values.

Each row is one state.  
Each column is one action.


In [82]:
np.set_printoptions(precision=3, suppress=True)

print("Learned Q-table:")
print(Q_values)

best_actions = np.argmax(Q_values, axis=1)

best_action_names = np.array([action_names[a] for a in best_actions]).reshape(4, 4)
print("best actions:\n", best_action_names.reshape(4, 4))

Learned Q-table:
[[0.735 0.774 0.774 0.735]
 [0.735 0.    0.815 0.774]
 [0.774 0.857 0.774 0.815]
 [0.815 0.    0.763 0.726]
 [0.774 0.815 0.    0.735]
 [0.    0.    0.    0.   ]
 [0.    0.902 0.    0.815]
 [0.    0.    0.    0.   ]
 [0.815 0.    0.857 0.774]
 [0.815 0.902 0.902 0.   ]
 [0.857 0.95  0.    0.857]
 [0.    0.    0.    0.   ]
 [0.    0.    0.    0.   ]
 [0.    0.902 0.95  0.857]
 [0.902 0.95  1.    0.902]
 [0.    0.    0.    0.   ]]
best actions:
 [['Down' 'Right' 'Down' 'Left']
 ['Down' 'Left' 'Down' 'Left']
 ['Right' 'Down' 'Down' 'Left']
 ['Left' 'Right' 'Right' 'Left']]


## 6. Test the Learned Policy

Now we use the greedy policy derived from the Q-table.

```python
action = np.argmax(Q_values[state])
```


In [84]:
import time

def show_one_episode_human(Q_values, seed=42, max_steps=100, delay=0.5):
    env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="human")

    state, info = env.reset(seed=seed)
  
    done = False
    total_reward = 0
    steps = 0
    path = [state]

    while not done and steps < max_steps:
        action = np.argmax(Q_values[state])
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        total_reward += reward
        steps += 1
        state = next_state
        path.append(state)

        time.sleep(delay)
    env.close()
    return total_reward, steps, path


test_return, test_steps, test_path = show_one_episode_human(
    Q_values, seed=42, max_steps=100, delay=0.2)



print("Test return:", test_return)
print("Test steps:", test_steps)
print("Path:", test_path)
print("Path as grid positions:", [(s // 4, s % 4) for s in test_path])

Test return: 1
Test steps: 6
Path: [0, 4, 8, 9, 13, 14, 15]
Path as grid positions: [(0, 0), (1, 0), (2, 0), (2, 1), (3, 1), (3, 2), (3, 3)]



## Discussion Questions

1. Why do we need future reward instead of only immediate reward?
2. What happens if \(\epsilon\) is too high?
3. What happens if \(\epsilon\) is too low?
4. What happens if \(\gamma=0\)?
5. Why does Q-table work for FrozenLake?
6. Why might Q-table fail for image-based games such as Atari or Mario?
7. How is Q-learning different from simply following the highest immediate reward?
8. What if you change 
    ```python
    is_slippery=False
    ```
    to:

    ```python
    is_slippery=True
    ```
9. When the environment becomes stochastic
    - Does training become harder?
    - Do you need more episodes?
    - Does the learned policy change?